In [1]:
from pathlib import Path
import random
from PIL import Image
import io
import matplotlib.pyplot as plt
import numpy as np
from entropy import *

data_folder = Path("../src/Dataset/test_data")
output_folder = Path("../src/Dataset/test_data_decoded")
output_folder.mkdir(parents=True, exist_ok=True)

def load_data(folder) -> list:
    data_rgb = []

    for file in folder.glob("*.png"):
        image = Image.open(file).convert("RGB")
        data_rgb.append((file, image.copy()))
        image.close()

    print("Count of loaded .png images:", len(data_rgb))
    return data_rgb


loaded_data_rgb = load_data(data_folder)

codec = JPEGCodec3(
    block_size=8,
    color_space="YCbCr",
    q_y=150.0,
    q_c=150.0 * 1.6,
)

for file, image in loaded_data_rgb:
    encoded = codec.encode_symbols(image)
    decoded = codec.decode_symbols(encoded)

    decoded.save(output_folder / file.name)

Count of loaded .png images: 71


In [6]:
import os
import random
from pathlib import Path

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms

from skimage.metrics import structural_similarity as ssim


# ============================================================
# 1. Reproducibility
# ============================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# ============================================================
# 2. Dataset
# ============================================================

class JPEGPairDataset(Dataset):
    """
    Dataset for training:

        decoded RGB image -> original RGB image

    It trains on random patches to reduce memory usage.
    """

    def __init__(
        self,
        original_dir="Data/original",
        decoded_dir="Data/decoded",
        patch_size=256,
        use_random_crop=True
    ):
        self.original_dir = Path(original_dir)
        self.decoded_dir = Path(decoded_dir)
        self.patch_size = patch_size
        self.use_random_crop = use_random_crop

        valid_extensions = [".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"]

        self.original_paths = sorted([
            p for p in self.original_dir.iterdir()
            if p.suffix.lower() in valid_extensions
        ])

        self.pairs = []

        for original_path in self.original_paths:
            decoded_path = self.decoded_dir / original_path.name

            if decoded_path.exists():
                self.pairs.append((original_path, decoded_path))
            else:
                print(f"Warning: no decoded match found for {original_path.name}")

        if len(self.pairs) == 0:
            raise RuntimeError("No matching image pairs found.")

        self.to_tensor = transforms.ToTensor()

    def __len__(self):
        return len(self.pairs)

    def random_crop_pair(self, original, decoded):
        width, height = original.size

        ps = self.patch_size

        if width < ps or height < ps:
            raise ValueError(
                f"Image is smaller than patch size. "
                f"Image size: {(width, height)}, patch size: {ps}"
            )

        left = random.randint(0, width - ps)
        top = random.randint(0, height - ps)

        original_patch = original.crop((left, top, left + ps, top + ps))
        decoded_patch = decoded.crop((left, top, left + ps, top + ps))

        return original_patch, decoded_patch

    def center_crop_pair(self, original, decoded):
        width, height = original.size

        ps = self.patch_size

        left = (width - ps) // 2
        top = (height - ps) // 2

        original_patch = original.crop((left, top, left + ps, top + ps))
        decoded_patch = decoded.crop((left, top, left + ps, top + ps))

        return original_patch, decoded_patch

    def __getitem__(self, idx):
        original_path, decoded_path = self.pairs[idx]

        original = Image.open(original_path).convert("RGB")
        decoded = Image.open(decoded_path).convert("RGB")

        if self.patch_size is not None:
            if self.use_random_crop:
                original, decoded = self.random_crop_pair(original, decoded)
            else:
                original, decoded = self.center_crop_pair(original, decoded)

        original = self.to_tensor(original)
        decoded = self.to_tensor(decoded)

        return {
            "decoded": decoded,
            "original": original,
            "filename": original_path.name
        }

# ============================================================
# 3. Residual CNN / ResNet model
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

class ResUNet(nn.Module):
    """
    Residual U-Net.
    """
    def __init__(self, in_channels: int = 1, out_channels: int = 1):
        super(ResUNet, self).__init__()

        class ResidualConvBlock(nn.Module):
            def __init__(self, in_ch, out_ch, kernel_size=3, stride=1, padding=1, dilation = 1):
                super(ResidualConvBlock, self).__init__()
                self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size, stride, padding, dilation = dilation, padding_mode='reflect')
                self.bn1 = nn.BatchNorm2d(out_ch)
                self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size, stride, padding, dilation = dilation, padding_mode='reflect')
                self.bn2 = nn.BatchNorm2d(out_ch)
                self.relu1 = nn.LeakyReLU( 0.2, inplace=True)
                self.dropout = nn.Dropout(p=0.2)
                self.relu2 = nn.LeakyReLU(0.2, inplace=True)
                self.shortcut = nn.Identity()
                if in_ch != out_ch:
                    self.shortcut = nn.Sequential(
                        nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=1, padding=0),
                        nn.BatchNorm2d(out_ch)
                    )
            
            def forward(self, x):
                identity = self.shortcut(x)
                out = self.conv1(x)
                out = self.bn1(out)
                out = self.relu1(out)
                out = self.conv2(out)
                out = self.bn2(out)

                out += identity  # Residual connection
                out = self.relu2(out)
                out = self.dropout(out)
                return out
        
        def conv_block(in_ch, out_ch, kernel = 3, stride = 1, dil = 1, pad = 1):
            return ResidualConvBlock(in_ch, out_ch, kernel_size=kernel, stride=stride, padding=pad, dilation = dil)

        class Up(nn.Module):
            def __init__(self, in_ch, out_ch, kernel = 3, pad = 1):
                super(Up, self).__init__()
                self.up = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)
                self.conv = conv_block(out_ch * 2, out_ch, kernel = kernel, pad = pad)

            def forward(self, x1, x2):
                x1 = self.up(x1)
                diffY = x2.size()[2] - x1.size()[2]
                diffX = x2.size()[3] - x1.size()[3]
                x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])
                x = torch.cat([x2, x1], dim=1)
                return self.conv(x)
        
        def down(in_ch, out_ch, kernel = 3, pad = 1, dil = 1):
            return nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=2),
                conv_block(in_ch, out_ch, kernel = kernel, pad = pad, dil = dil)
            )

        # Encoder layers
        self.inc = conv_block(in_channels, 64, kernel = 13, dil = 2, pad = 12)
        self.down1 = down(64, 128, kernel = 7, pad = 3)     # 512x672 -> 256x336    
        self.down2 = down(128, 256, kernel = 5, pad = 2)    # 256x336 -> 128x168
        self.down3 = down(256, 512)                         # 128x168 -> 64x84
        self.down4 = down(512, 1024)                        # 64x84 -> 32x42             

        # Decoder layers
        self.up1 = Up(1024, 512)                            # 32x42 -> 64x84
        self.up2 = Up(512, 256)                             # 64x84 -> 128x168
        self.up3 = Up(256, 128, kernel = 5, pad = 2)        # 128x168 -> 256x336
        self.up4 = Up(128, 64, kernel = 7, pad = 3)         # 256x336 -> 512x672
        # Final output layer
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        pad = 32  
       
        decoded_input = x  # Only used if wanted to be added at the end

        x = F.pad(x, (pad, pad, pad, pad), mode='reflect')

        # Encoder
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        # Decoder
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)

        # Final output
        residual  = self.outc(x)
        # Crop the padding
        residual  = residual [..., pad:-pad, pad:-pad]

        enhanced = decoded_input + residual
        enhanced = torch.clamp(enhanced, 0.0, 1.0)
        return enhanced



# ============================================================
# 4. Loss functions
# ============================================================

class SobelEdgeLoss(nn.Module):
    """
    Edge loss to preserve edges and reduce visual artifacts.
    """

    def __init__(self, channels=1):
        super().__init__()

        sobel_x = torch.tensor(
            [[-1, 0, 1],
             [-2, 0, 2],
             [-1, 0, 1]],
            dtype=torch.float32
        )

        sobel_y = torch.tensor(
            [[-1, -2, -1],
             [0, 0, 0],
             [1, 2, 1]],
            dtype=torch.float32
        )

        sobel_x = sobel_x.view(1, 1, 3, 3).repeat(channels, 1, 1, 1)
        sobel_y = sobel_y.view(1, 1, 3, 3).repeat(channels, 1, 1, 1)

        self.register_buffer("sobel_x", sobel_x)
        self.register_buffer("sobel_y", sobel_y)

        self.channels = channels

    def forward(self, pred, target):
        pred_dx = F.conv2d(
            pred,
            self.sobel_x,
            padding=1,
            groups=self.channels
        )

        pred_dy = F.conv2d(
            pred,
            self.sobel_y,
            padding=1,
            groups=self.channels
        )

        target_dx = F.conv2d(
            target,
            self.sobel_x,
            padding=1,
            groups=self.channels
        )

        target_dy = F.conv2d(
            target,
            self.sobel_y,
            padding=1,
            groups=self.channels
        )

        pred_edges = torch.sqrt(pred_dx ** 2 + pred_dy ** 2 + 1e-8)
        target_edges = torch.sqrt(target_dx ** 2 + target_dy ** 2 + 1e-8)

        return F.l1_loss(pred_edges, target_edges)


class ReconstructionLoss(nn.Module):
    """
    Combined loss:

        total = MSE + lambda_mae * MAE + lambda_edge * EdgeLoss
    """

    def __init__(
        self,
        channels=1,
        lambda_mse=1.0,
        lambda_mae=0.2,
        lambda_edge=0.05
    ):
        super().__init__()

        self.lambda_mse = lambda_mse
        self.lambda_mae = lambda_mae
        self.lambda_edge = lambda_edge

        self.mse = nn.MSELoss()
        self.mae = nn.L1Loss()
        self.edge = SobelEdgeLoss(channels=channels)

    def forward(self, enhanced, original):
        loss_mse = self.mse(enhanced, original)
        loss_mae = self.mae(enhanced, original)
        loss_edge = self.edge(enhanced, original)

        total_loss = (
            self.lambda_mse * loss_mse
            + self.lambda_mae * loss_mae
            + self.lambda_edge * loss_edge
        )

        return total_loss, {
            "mse": loss_mse.item(),
            "mae": loss_mae.item(),
            "edge": loss_edge.item(),
            "total": total_loss.item()
        }


# ============================================================
# 5. Metrics
# ============================================================

def tensor_to_numpy_image(tensor):
    """
    Converts a tensor image [C, H, W] in [0, 1]
    to numpy image [H, W] or [H, W, C] in [0, 255].
    """

    tensor = tensor.detach().cpu().clamp(0.0, 1.0)

    image = tensor.numpy()

    if image.shape[0] == 1:
        image = image[0]
    else:
        image = np.transpose(image, (1, 2, 0))

    image = image * 255.0

    return image.astype(np.float64)


def mse_np(img1, img2):
    return np.mean((img1.astype(np.float64) - img2.astype(np.float64)) ** 2)


def psnr_np(img1, img2, max_val=255.0):
    mse = mse_np(img1, img2)

    if mse == 0:
        return float("inf")

    return 10 * np.log10((max_val ** 2) / mse)


def ssim_np(img1, img2):
    if img1.ndim == 2:
        return ssim(img1, img2, data_range=255)

    return ssim(
        img1,
        img2,
        data_range=255,
        channel_axis=2
    )


@torch.no_grad()
def compute_metrics_for_loader(model, dataloader, device):
    model.eval()

    normal_psnr_values = []
    enhanced_psnr_values = []

    normal_ssim_values = []
    enhanced_ssim_values = []

    normal_mse_values = []
    enhanced_mse_values = []

    for batch in dataloader:
        decoded = batch["decoded"].to(device)
        original = batch["original"].to(device)

        enhanced = model(decoded)

        batch_size = decoded.size(0)

        for i in range(batch_size):
            decoded_np = tensor_to_numpy_image(decoded[i])
            original_np = tensor_to_numpy_image(original[i])
            enhanced_np = tensor_to_numpy_image(enhanced[i])

            normal_mse_values.append(mse_np(original_np, decoded_np))
            enhanced_mse_values.append(mse_np(original_np, enhanced_np))

            normal_psnr_values.append(psnr_np(original_np, decoded_np))
            enhanced_psnr_values.append(psnr_np(original_np, enhanced_np))

            normal_ssim_values.append(ssim_np(original_np, decoded_np))
            enhanced_ssim_values.append(ssim_np(original_np, enhanced_np))

    return {
        "normal_mse": float(np.mean(normal_mse_values)),
        "enhanced_mse": float(np.mean(enhanced_mse_values)),

        "normal_psnr": float(np.mean(normal_psnr_values)),
        "enhanced_psnr": float(np.mean(enhanced_psnr_values)),

        "normal_ssim": float(np.mean(normal_ssim_values)),
        "enhanced_ssim": float(np.mean(enhanced_ssim_values)),
    }


# ============================================================
# 6. Training and validation
# ============================================================

def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()

    total_loss = 0.0
    total_mse = 0.0
    total_mae = 0.0
    total_edge = 0.0

    for batch in dataloader:
        decoded = batch["decoded"].to(device)
        original = batch["original"].to(device)

        optimizer.zero_grad()

        enhanced = model(decoded)

        loss, loss_dict = criterion(enhanced, original)

        loss.backward()
        optimizer.step()

        total_loss += loss_dict["total"]
        total_mse += loss_dict["mse"]
        total_mae += loss_dict["mae"]
        total_edge += loss_dict["edge"]

    n = len(dataloader)

    return {
        "loss": total_loss / n,
        "mse": total_mse / n,
        "mae": total_mae / n,
        "edge": total_edge / n
    }


@torch.no_grad()
def validate_one_epoch(model, dataloader, criterion, device):
    model.eval()

    total_loss = 0.0
    total_mse = 0.0
    total_mae = 0.0
    total_edge = 0.0

    for batch in dataloader:
        decoded = batch["decoded"].to(device)
        original = batch["original"].to(device)

        enhanced = model(decoded)

        loss, loss_dict = criterion(enhanced, original)

        total_loss += loss_dict["total"]
        total_mse += loss_dict["mse"]
        total_mae += loss_dict["mae"]
        total_edge += loss_dict["edge"]

    n = len(dataloader)

    return {
        "loss": total_loss / n,
        "mse": total_mse / n,
        "mae": total_mae / n,
        "edge": total_edge / n
    }


# ============================================================
# 7. Save example enhanced images
# ============================================================

@torch.no_grad()
def save_example_outputs(model, dataloader, device, output_dir="outputs", max_images=8):
    model.eval()

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    saved = 0

    for batch in dataloader:
        decoded = batch["decoded"].to(device)
        original = batch["original"].to(device)
        filenames = batch["filename"]

        enhanced = model(decoded)

        batch_size = decoded.size(0)

        for i in range(batch_size):
            if saved >= max_images:
                return

            decoded_np = tensor_to_numpy_image(decoded[i])
            original_np = tensor_to_numpy_image(original[i])
            enhanced_np = tensor_to_numpy_image(enhanced[i])

            filename_stem = Path(filenames[i]).stem

            Image.fromarray(decoded_np.astype(np.uint8)).save(
                output_dir / f"{filename_stem}_decoded.png"
            )

            Image.fromarray(original_np.astype(np.uint8)).save(
                output_dir / f"{filename_stem}_original.png"
            )

            Image.fromarray(enhanced_np.astype(np.uint8)).save(
                output_dir / f"{filename_stem}_enhanced.png"
            )

            saved += 1

# ============================================================
# 8. Main script
# ============================================================

def main():
    set_seed(42)

    # -----------------------------
    # Settings
    # -----------------------------

    original_dir = "../src/Dataset/test_data"
    decoded_dir = "../src/Dataset/test_data_decoded"

    grayscale = False
    image_size = None

    batch_size = 4
    num_epochs = 10
    learning_rate = 1e-4

    train_ratio = 0.70
    val_ratio = 0.15
    test_ratio = 0.15

    num_features = 64
    num_residual_blocks = 8

    save_dir = Path("checkpoints")
    save_dir.mkdir(parents=True, exist_ok=True)

    output_dir = "outputs"

    # -----------------------------
    # Device
    # -----------------------------

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # -----------------------------
    # Dataset
    # -----------------------------

    dataset = JPEGPairDataset(
        original_dir=original_dir,
        decoded_dir=decoded_dir,
        patch_size=256,
        use_random_crop=True
    )

    print(f"Total matched image pairs: {len(dataset)}")

    total_size = len(dataset)

    train_size = int(train_ratio * total_size)
    val_size = int(val_ratio * total_size)
    test_size = total_size - train_size - val_size

    train_dataset, val_dataset, test_dataset = random_split(
        dataset,
        [train_size, val_size, test_size],
        generator=torch.Generator().manual_seed(42)
    )

    print(f"Train size: {len(train_dataset)}")
    print(f"Val size:   {len(val_dataset)}")
    print(f"Test size:  {len(test_dataset)}")

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0
    )

    # -----------------------------
    # Model
    # -----------------------------

    in_channels = 1 if grayscale else 3

    model = ResUNet(in_channels=3, out_channels=3).to(device)

    criterion = ReconstructionLoss(
        channels=in_channels,
        lambda_mse=1.0,
        lambda_mae=0.2,
        lambda_edge=0.05
    )

    optimizer = optim.Adam(
        model.parameters(),
        lr=learning_rate
    )

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=5
    )

    # -----------------------------
    # Training
    # -----------------------------

    best_val_loss = float("inf")

    for epoch in range(num_epochs):
        train_stats = train_one_epoch(
            model=model,
            dataloader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            device=device
        )

        val_stats = validate_one_epoch(
            model=model,
            dataloader=val_loader,
            criterion=criterion,
            device=device
        )

        scheduler.step(val_stats["loss"])

        print(
            f"Epoch [{epoch + 1:03d}/{num_epochs}] "
            f"Train Loss: {train_stats['loss']:.6f} "
            f"Val Loss: {val_stats['loss']:.6f} "
            f"Train MSE: {train_stats['mse']:.6f} "
            f"Val MSE: {val_stats['mse']:.6f}"
        )

        if val_stats["loss"] < best_val_loss:
            best_val_loss = val_stats["loss"]

            torch.save(
                {
                    "epoch": epoch + 1,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "val_loss": best_val_loss,
                    "grayscale": grayscale,
                    "num_features": num_features,
                    "num_residual_blocks": num_residual_blocks
                },
                save_dir / "best_residual_cnn.pth"
            )

            print("Saved new best model.")

    # -----------------------------
    # Load best model
    # -----------------------------

    checkpoint = torch.load(
        save_dir / "best_residual_cnn.pth",
        map_location=device
    )

    model.load_state_dict(checkpoint["model_state_dict"])

    # -----------------------------
    # Final test metrics
    # -----------------------------

    test_metrics = compute_metrics_for_loader(
        model=model,
        dataloader=test_loader,
        device=device
    )

    print("\nFinal test results")
    print("------------------")
    print(f"Normal decoded MSE:      {test_metrics['normal_mse']:.4f}")
    print(f"CNN enhanced MSE:        {test_metrics['enhanced_mse']:.4f}")
    print()
    print(f"Normal decoded PSNR:     {test_metrics['normal_psnr']:.4f} dB")
    print(f"CNN enhanced PSNR:       {test_metrics['enhanced_psnr']:.4f} dB")
    print()
    print(f"Normal decoded SSIM:     {test_metrics['normal_ssim']:.4f}")
    print(f"CNN enhanced SSIM:       {test_metrics['enhanced_ssim']:.4f}")

    # -----------------------------
    # Save visual examples
    # -----------------------------

    save_example_outputs(
        model=model,
        dataloader=test_loader,
        device=device,
        output_dir=output_dir,
        max_images=8
    )

    print(f"\nSaved example outputs to: {output_dir}")


if __name__ == "__main__":
    main()

Using device: cpu
Total matched image pairs: 71
Train size: 49
Val size:   10
Test size:  12


KeyboardInterrupt: 